In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv('D:\\PycharmProjects\\mf-ml\\feature_selection_output\\ml_feature_dataset.csv')

# Selected feature set based on Wu et al. (2025) feature selection protocol
selected_features = [
    # Fundamentals
    'net_flow_percent',
    # Performance Momentum (Alphas & t-stats)
    'const_90', 't_const_90',
    'const_120', 't_const_120',
    'const_180', 't_const_180',
    'const_250', 't_const_250',
    # Risk-Adjusted Ratios
    'sr_p1y', 'ir_p1y',
    'sr_p3y', 'ir_p3y'
]

# Create modeling feature matrix
X = df[selected_features]
y = df['outperforms_benchmark']

In [4]:
import pandas as pd
import scipy.stats as stats

# 1. Load the dataset
#df = pd.read_csv('ml_feature_dataset.csv')
df = pd.read_csv('D:\\PycharmProjects\\mf-ml\\feature_selection_output\\ml_feature_dataset.csv')

# 2. Define target and metadata columns to exclude from features
non_feature_cols = ['scheme_name', 'month_date', 'target_month', 'outperforms_benchmark', 'monthly_return_percent']
feature_cols = [c for c in df.columns if c not in non_feature_cols]

# 3. Compute cross-sectional Spearman IC for each month and feature
results = []

for feature in feature_cols:
    monthly_ics = []

    for month, group in df.groupby('month_date'):
        # Pairwise drop of missing values for the feature and forward return
        valid_data = group[[feature, 'monthly_return_percent']].dropna()

        # Require a minimum sample of funds per month for valid rank correlation
        if len(valid_data) >= 10:
            ic, _ = stats.spearmanr(valid_data[feature], valid_data['monthly_return_percent'])
            if not pd.isna(ic):
                monthly_ics.append(ic)

    if monthly_ics:
        mean_ic = float(pd.Series(monthly_ics).mean())
        std_ic = float(pd.Series(monthly_ics).std())
        ic_ir = mean_ic / std_ic if std_ic != 0 else 0.0  # Information Ratio of IC

        results.append({
            'Feature': feature,
            'Mean_IC': round(mean_ic, 4),
            'Abs_Mean_IC': round(abs(mean_ic), 4),
            'IC_Std': round(std_ic, 4),
            'IC_IR': round(ic_ir, 4),
            'Passes_Threshold (|IC| >= 0.03)': abs(mean_ic) >= 0.03
        })

# 4. Display sorted results DataFrame
ic_summary = pd.DataFrame(results).sort_values(by='Abs_Mean_IC', ascending=False).reset_index(drop=True)
print(ic_summary.to_string())

             Feature  Mean_IC  Abs_Mean_IC  IC_Std   IC_IR  Passes_Threshold (|IC| >= 0.03)
0           const_90   0.3749       0.3749  0.2491  1.5052                             True
1         t_const_90   0.3705       0.3705  0.2455  1.5094                             True
2             ir_p6m   0.3683       0.3683  0.2818  1.3070                             True
3             sr_p6m   0.3654       0.3654  0.2811  1.3000                             True
4          const_120   0.3232       0.3232  0.2817  1.1476                             True
5        t_const_120   0.3230       0.3230  0.2678  1.2059                             True
6        t_const_180   0.2951       0.2951  0.2574  1.1465                             True
7             sr_p1y   0.2775       0.2775  0.2919  0.9507                             True
8          const_180   0.2772       0.2772  0.2719  1.0196                             True
9             ir_p1y   0.2725       0.2725  0.2964  0.9192                      

In [6]:
model_df = df.copy()

print("model_df shape:", model_df.shape)

print("\nColumns:")
for i, col in enumerate(model_df.columns, 1):
    print(f"{i:2d}. {col}")

model_df shape: (7034, 29)

Columns:
 1. scheme_name
 2. month_date
 3. target_month
 4. outperforms_benchmark
 5. t_hml_250
 6. t_hml_750
 7. t_hml_180
 8. sr_p1y
 9. ir_p1y
10. ir_p6m
11. monthly_return_percent
12. t_const_180
13. sr_p6m
14. t_const_250
15. ir_p3y
16. const_90
17. const_180
18. t_const_90
19. t_const_120
20. const_250
21. t_hml_120
22. const_120
23. t_mkt_750
24. t_umd_750
25. t_hml_90
26. sr_p3y
27. r2_750
28. net_flow_percent
29. t_smb_250
